In [25]:
import os
import random
import shutil
from sklearn.model_selection import train_test_split

# Verzeichnisse definieren


### Splitting the LCT dataset into training and test data

In [26]:
input_dir = 'lct_txt'
output_dir = 'lct_p1'
dataset_dir = 'dataset'
train_input_dir = os.path.join(dataset_dir, 'train', 'input')
train_output_dir = os.path.join(dataset_dir, 'train', 'output')
test_input_dir = os.path.join(dataset_dir, 'test', 'input')
test_output_dir = os.path.join(dataset_dir, 'test', 'output')

# Verzeichnisstruktur erstellen
os.makedirs(train_input_dir, exist_ok=True)
os.makedirs(train_output_dir, exist_ok=True)
os.makedirs(test_input_dir, exist_ok=True)
os.makedirs(test_output_dir, exist_ok=True)

# Dateien sammeln
input_files = sorted(os.listdir(input_dir))
output_files = sorted(os.listdir(output_dir))

# Überprüfen, ob alle Dateien ein passendes Gegenstück haben
input_files = [f for f in input_files if f in output_files]

# Dateien splitten (80% Training, 20% Test)
train_files, test_files = train_test_split(input_files, test_size=0.2, random_state=42)

print(len(train_files))
# Dateien kopieren
def copy_files(file_list, src_dir, dest_dir):
    for file_name in file_list:
        shutil.copy(os.path.join(src_dir, file_name), os.path.join(dest_dir, file_name))

# Trainingsdaten kopieren
copy_files(train_files, input_dir, train_input_dir)
copy_files(train_files, output_dir, train_output_dir)

# Testdaten kopieren
copy_files(test_files, input_dir, test_input_dir)
copy_files(test_files, output_dir, test_output_dir)
print(f"Dataset creation completed. Training files: {len(train_files)}, Test files: {len(test_files)}")

804
Dataset creation completed. Training files: 804, Test files: 202


### Create Dataset

In [27]:
instruction_text ="""Create an edited version of the following inclusion and exclusion criteria that incorporates the logical operators [AND], [OR], [NOT].
Apply these rules for inserting the logical operators:
[OR]:
- Insert [OR] to indicate alternatives, where at least one of the conditions must be satisfied.
- Always write an [OR] before an "or", "and/or", "and / or" in the text.
- Example: Patients with delirium [OR] or depression. Cancer [OR] and / or HIV. Illness, [OR] or injury.
- Write [OR] after a comma if it logically separates the sentence with an [OR].
- Example: Illness, [OR] or injury. Type A, [OR] B.
- Write an [OR] after "/" when they indicate alternatives.
- Example: Type A/ [OR] B.
- Exception: If examples or instances of an Event follow in parentheses, [OR] should not be annotated.
[AND]:
- Insert [AND] to join two  sentences.
- Write an [AND] before words like "with", "who", "in addition", "plus", "and", "but", "that", "despite", "having" when it is appropriate.
- Example: Stroke [AND] with first-ever aphasic symptoms. Patients [AND] who are postmenopausal.
- Write an [AND] before "and" when it joins two or more conditions or criteria that must all be met for eligibility.
- Example: Postmenopausal [AND] and preferabley on hormone replacement therapy.
- Exception: When [AND] is used in an Equality Comparison (i.e., between two numeric values) it should not be annotated.
[NOT]:
- Insert [NOT] before a condition to exclude or negate it.
- Always write a [NOT] before "no", "not", "none".
- Write a [NOT] before phrases like "don't", "free", "prevent", "Inability", "lack", "impossible", "off", "without", "unable", "naive", "excluded", "absence" when they negate a condition or requirement.
- Example: Patients [NOT] unable to take medicines. [NOT] No history of cancer. [NOT] Not on hormone replacement therapy. [NOT] don't  have a history of smoking. [NOT] free from a history of alcoholism.

General rules:
- The operators [OR], [AND], [NOT] should be written before the words like or, and, not, instead of replacing them
- An [AND], [OR], [NOT] must not come before a line segment (\n).
- Please carefully read through the given criteria text, then output only an edited version with the logical operators incorporated, enclosed in <EDITED_CRITERIA> tags.
- Do not include any other explanations or output."""

In [28]:
import os
import pandas as pd
from datasets import Dataset, DatasetDict

# Verzeichnisse definieren
train_input_dir = 'dataset/train/input'
train_output_dir = 'dataset/train/output'
test_input_dir = 'dataset/test/input'
test_output_dir = 'dataset/test/output'

# Funktion zum Lesen der Dateien und Erstellen eines DataFrames
def create_dataframe(input_dir, output_dir):
    data = []
    for file_name in os.listdir(input_dir):
        if file_name in os.listdir(output_dir):
            with open(os.path.join(input_dir, file_name), 'r', encoding='utf-8') as f_in, \
                    open(os.path.join(output_dir, file_name), 'r', encoding='utf-8') as f_out:
                input_text = f_in.read().strip()
                output_text = f_out.read().strip()
                nct_number = os.path.splitext(file_name)[0]
                data.append({"input": input_text, "output": output_text, "nct_number": nct_number})
    return pd.DataFrame(data)

# DataFrames für Training und Test erstellen
train_df = create_dataframe(train_input_dir, train_output_dir)
test_df = create_dataframe(test_input_dir, test_output_dir)



# Instruction Text hinzufügen
train_df['instruction'] = instruction_text
test_df['instruction'] = instruction_text

# NCT-Nummer an die DataFrames anhängen
train_df = train_df[['input', 'output', 'instruction']] # , 'nct_number'
test_df = test_df[['input', 'output', 'instruction']] # , 'nct_number'

# Pandas DataFrames in Hugging Face Datasets umwandeln
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# DatasetDict erstellen
dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

# Hugging Face Datasets speichern
dataset_dict.save_to_disk('dataset/lct_dataset')

print("Datasets wurden erfolgreich erstellt und gespeichert.")


Saving the dataset (0/1 shards):   0%|          | 0/804 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/202 [00:00<?, ? examples/s]

Datasets wurden erfolgreich erstellt und gespeichert.


In [30]:
import os
from datasets import load_from_disk
dataset_path = 'dataset/lct_dataset_v2'
dataset_dict = load_from_disk(dataset_path)

In [31]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction', 'nct_number'],
        num_rows: 804
    })
    test: Dataset({
        features: ['input', 'output', 'instruction', 'nct_number'],
        num_rows: 202
    })
})

In [32]:
print("Train Dataset:")
print(dataset_dict['train'][0])

Train Dataset:
{'input': 'Inclusion Criteria:\n  1. inflammatory bowel disease\n  2. on methotrexate at appropriate dosing\n  3. normal folate levels at onset of study\n  4. treatment with folic acid\n  5. ages 2-21 years\nExclusion Criteria:\n  1. abnormal folate levels\n  2. age > 21 or less than 2', 'output': 'Inclusion Criteria:\n 1. inflammatory bowel disease\n 2. on methotrexate at appropriate dosing\n 3. normal folate levels at onset of study\n 4. treatment with folic acid\n 5. ages 2-21 years\nExclusion Criteria:\n 1. abnormal folate levels\n 2. age > 21 [OR] or less than 2', 'instruction': 'Create an edited version of the following inclusion and exclusion criteria that incorporates the logical operators [AND], [OR], [NOT].\nApply these rules for inserting the logical operators:\n[OR]:\n- Insert [OR] to indicate alternatives, where at least one of the conditions must be satisfied.\n- Always write an [OR] before an "or", "and/or", "and / or" in the text.\n- Example: Patients wit

In [33]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

#EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) #+ EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

In [34]:
from datasets import load_dataset
dataset = load_dataset("yahma/alpaca-cleaned", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [37]:
dataset[0]

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'input': '',
 'instruction': 'Give three tips for staying healthy.',
 'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes th

In [38]:
import os
from datasets import load_from_disk
dataset_path = 'dataset/lct_dataset_v2'
dataset = load_from_disk(dataset_path)
dataset = dataset['train']
dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/804 [00:00<?, ? examples/s]